# V28 - Non-Building Subgraph GNN

## v25 → v28 변경사항
- **v25**: 10,000노드 full grid → GT edge 비율 0.025% → class imbalance
- **v28**: 건물 제외 free pixel만 노드 → ~5,000노드, GT edge 비율 ~0.5% (20배 개선)

## GT 파이프라인
- 4방향 gate × 4 cost function(Dijkstra×3 + A*) = 16 variant → pixel vote → consensus
- 49개 캠퍼스 수동 선택 → `collegemap/gt_masks_final/` (c4/c8/c12 per campus)
- 학습 시 GT 증강 없음 (campus당 1개 고정 GT)

## v25에서 그대로 유지
- gate_nodes + cluster_nodes → terminals 구조
- AMP safe 학습 루프 (gradient accumulation)
- BCE + Dice + FP ridge 손실
- Train/Val/Test 분리 (49 osm = train+val, 나머지 = test)

In [1]:
import os
!git -C swe3032 pull 2>/dev/null || git clone https://github.com/SKKUAIProjectTeam1/swe3032.git
os.chdir('swe3032')

Cloning into 'swe3032'...
remote: Enumerating objects: 1301, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 1301 (delta 98), reused 119 (delta 71), pack-reused 1150 (from 1)
Receiving objects: 100% (1301/1301), 204.23 MiB | 24.81 MiB/s, done.
Resolving deltas: 100% (340/340), done.
Updating files: 100% (1035/1035), done.


In [ ]:
import glob, os, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from scipy.ndimage import distance_transform_edt, maximum_filter, gaussian_filter

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
CODE_VERSION  = 'v28_subgraph_gnn'
print(f'코드 버전: {CODE_VERSION}')
RES           = 100
N             = RES * RES
NODE_DIM      = 7   # ridge, dist_n, x, y, cluster_indicator, dist_to_cluster, dist_to_gate
GAT_HEADS     = 4
EPOCHS        = 220
LR            = 4e-4
WARMUP_EPOCHS = 20
ATTN_DROPOUT  = 0.05
ACCUM_STEPS   = 10
N_GATES       = 2
GATE_MIN_DIST = 14
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLUSTER_EPS       = 13.0
ROAD_CLEAR_MIN    = 3.0
ROAD_CLEAR_MAX    = 24.0
RIDGE_FILTER_SIZE = 7
EDGE_THR          = 0.50
PRUNE_ITERS       = 12

IMG_DIR      = 'collegemap/images'
TXT_DIR      = 'collegemap/txt'
GT_FINAL_DIR = 'collegemap/gt_masks_final'
OUT_DIR      = 'output'
CKPT         = os.path.join(OUT_DIR, 'v28_subgraph_gnn.pth')
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
# ── 공통 헬퍼 (v25와 동일) ──────────────────────────────────────────────────────
def _nearest_free_node(cy, cx, is_bld_grid):
    non_bld_yx = np.argwhere(is_bld_grid == 0)
    if len(non_bld_yx) == 0: return int(cy)*RES + int(cx)
    dists = (non_bld_yx[:,0]-cy)**2 + (non_bld_yx[:,1]-cx)**2
    best  = non_bld_yx[np.argmin(dists)]
    return int(best[0])*RES + int(best[1])

def _make_common_road_ridge(is_bld_grid):
    free  = is_bld_grid == 0
    dist  = distance_transform_edt(free)
    clear = (dist > ROAD_CLEAR_MIN) & (dist < ROAD_CLEAR_MAX)
    campus_influence = gaussian_filter(is_bld_grid.astype(np.float32), sigma=7.0)
    near_campus = campus_influence > 0.01
    local_max   = dist == maximum_filter(dist, size=RIDGE_FILTER_SIZE)
    ridge       = free & clear & near_campus & local_max
    ridge_score = gaussian_filter(ridge.astype(np.float32), sigma=1.2)
    if ridge_score.max() < 1e-6:
        band = free & clear & near_campus
        ridge_score = gaussian_filter(band.astype(np.float32), sigma=1.0)
    ridge_score = ridge_score / (ridge_score.max() + 1e-6)
    return ridge_score.astype(np.float32), dist.astype(np.float32)

def _cluster_centers(centers, eps=CLUSTER_EPS):
    n=len(centers); visited=[False]*n; clusters=[]
    for i in range(n):
        if visited[i]: continue
        stack=[i]; visited[i]=True; cluster=[]
        while stack:
            u=stack.pop(); cluster.append(u); uy,ux=centers[u]
            for v in range(n):
                if visited[v]: continue
                vy,vx=centers[v]
                if ((uy-vy)**2+(ux-vx)**2)**0.5 <= eps:
                    visited[v]=True; stack.append(v)
        clusters.append(cluster)
    return clusters

def _nearest_ridge_node(cy, cx, is_bld_grid, ridge_grid):
    candidates = np.argwhere((is_bld_grid==0) & (ridge_grid>0.05))
    if len(candidates) == 0:
        return _nearest_free_node(cy, cx, is_bld_grid)
    dists = (candidates[:,0]-cy)**2 + (candidates[:,1]-cx)**2
    best  = candidates[np.argmin(dists)]
    return int(best[0])*RES + int(best[1])

# v25와 동일: 경계 진입점(gate) 선택
_BOUNDARY = sorted(set(
    [i for i in range(10, RES-10)] +
    [(RES-1)*RES+i for i in range(10, RES-10)] +
    [i*RES for i in range(10, RES-10)] +
    [i*RES+(RES-1) for i in range(10, RES-10)]
))

def _select_gate_nodes(cluster_nodes, is_bld_grid, ridge_grid):
    """v25 그대로: 캠퍼스 중심 방향 경계 비건물 노드 N_GATES개 선택."""
    if len(cluster_nodes) == 0: return []
    cy = np.mean([n//RES for n in cluster_nodes])
    cx = np.mean([n%RES  for n in cluster_nodes])
    candidates = []
    for node in _BOUNDARY:
        y, x = divmod(int(node), RES)
        if is_bld_grid[y, x] > 0: continue
        score = ((y-cy)**2+(x-cx)**2)**0.5 - ridge_grid[y,x]*8.0
        candidates.append((score, int(node)))
    candidates.sort(key=lambda t: t[0])
    chosen = []
    for _, node in candidates:
        y, x = divmod(node, RES)
        if all(abs(x-(p%RES))+abs(y-(p//RES)) >= GATE_MIN_DIST for p in chosen):
            chosen.append(node)
        if len(chosen) == N_GATES: break
    return chosen

In [ ]:
# ── 서브그래프 구성 (v28 신규) ──────────────────────────────────────────────────
def build_subgraph(is_bld_grid):
    """건물 픽셀을 제외한 free pixel만으로 구성된 그래프.

    v25 full grid (10000노드) 대비:
    - 노드: ~5000개 (건물 비율만큼 감소)
    - GT edge 비율: 0.025% → ~0.5% (class imbalance 20배 개선)

    Returns:
        free_pixels   (n_free,)  픽셀 인덱스 [0..N-1]
        pixel_to_node (N,)       건물=-1, free=서브그래프 노드번호
        sub_ei        (2, E)     서브그래프 edge_index (torch.long, CPU)
    """
    is_bld_flat = is_bld_grid.flatten()
    free_pixels = np.where(is_bld_flat == 0)[0].astype(np.int32)
    n_free = len(free_pixels)

    pixel_to_node = np.full(N, -1, dtype=np.int32)
    pixel_to_node[free_pixels] = np.arange(n_free, dtype=np.int32)

    rows, cols = [], []
    for px in free_pixels:
        y, x = divmod(int(px), RES)
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                if dy == 0 and dx == 0: continue
                ny, nx = y+dy, x+dx
                if 0 <= ny < RES and 0 <= nx < RES:
                    nb_node = pixel_to_node[ny*RES+nx]
                    if nb_node >= 0:
                        rows.append(int(pixel_to_node[px]))
                        cols.append(int(nb_node))

    return free_pixels, pixel_to_node, torch.tensor([rows, cols], dtype=torch.long)


def _make_sub_gt_edge_mask(gt_pixel_arr, free_pixels, sub_ei):
    """bool 100×100 배열 → 서브그래프 edge mask."""
    gt_flat = gt_pixel_arr.flatten()
    src_px  = free_pixels[sub_ei[0].numpy()]
    dst_px  = free_pixels[sub_ei[1].numpy()]
    return torch.tensor(gt_flat[src_px] & gt_flat[dst_px],
                        dtype=torch.float32, device=DEVICE)

In [ ]:
# ── 데이터 로딩 ─────────────────────────────────────────────────────────────────
def _find_txt(img_path):
    stem=os.path.basename(img_path).replace('_building_mask.png','')
    direct=os.path.join(TXT_DIR,stem+'_building_places.txt')
    if os.path.exists(direct): return direct
    prefix=stem.replace('-','_').split('_')[0]
    for fn in os.listdir(TXT_DIR):
        if fn.endswith('_building_places.txt') and fn.startswith(prefix):
            return os.path.join(TXT_DIR,fn)
    return None

def _dist_to_px(query_pixels, target_pixels, tau):
    """query_pixels의 각 픽셀에서 target_pixels까지 최소 거리의 exp decay."""
    if len(target_pixels) == 0:
        return np.zeros(len(query_pixels), dtype=np.float32)
    qy = query_pixels // RES;  qx = query_pixels % RES
    feats = []
    for t in target_pixels:
        ty, tx = divmod(int(t), RES)
        feats.append(np.sqrt((qy-ty)**2 + (qx-tx)**2))
    dmin = np.stack(feats, axis=0).min(axis=0)
    return np.exp(-dmin / tau).astype(np.float32)


def load_campus(img_path, txt_path):
    img    = Image.open(img_path).convert('L')
    W, H   = img.size
    is_bld = (np.array(img.resize((RES,RES),resample=Image.NEAREST)) > 128).astype(np.float32)

    ridge, dist = _make_common_road_ridge(is_bld)
    dist_n = (dist / (dist.max()+1e-6)).astype(np.float32)

    ns={}; exec(open(txt_path,encoding='utf-8').read(),ns)
    poly=ns['BUILDING_POLY']; building_ids=list(poly.keys())
    centers=[]
    for bid in building_ids:
        pts=poly[bid]
        cy=np.mean([p[1] for p in pts])*RES/H
        cx=np.mean([p[0] for p in pts])*RES/W
        centers.append((cy,cx))

    bld_nodes=[_nearest_free_node(cy,cx,is_bld) for cy,cx in centers]
    clusters=_cluster_centers(centers)
    cluster_nodes,cluster_groups=[],[]
    for cl in clusters:
        cy=np.mean([centers[i][0] for i in cl])
        cx=np.mean([centers[i][1] for i in cl])
        node=_nearest_ridge_node(cy,cx,is_bld,ridge)
        if node not in cluster_nodes:
            cluster_nodes.append(node)
            cluster_groups.append([building_ids[i] for i in cl])
    if len(cluster_nodes)<=1 and len(bld_nodes)>1:
        cluster_nodes=list(dict.fromkeys(bld_nodes))
        cluster_groups=[[bid] for bid in building_ids]

    gate_nodes = _select_gate_nodes(cluster_nodes, is_bld, ridge)
    terminals  = list(dict.fromkeys(gate_nodes + cluster_nodes))

    # ── 서브그래프 구성 (v28) ──────────────────────────────────────────────────
    free_pixels, pixel_to_node, sub_ei = build_subgraph(is_bld)
    n_free = len(free_pixels)

    def px_to_sub(px_list):
        return [int(pixel_to_node[p]) for p in px_list if pixel_to_node[p] >= 0]

    cluster_sub  = px_to_sub(cluster_nodes)
    gate_sub     = px_to_sub(gate_nodes)
    terminal_sub = px_to_sub(terminals)

    # ── GT 로드 (gt_masks_final/) ─────────────────────────────────────────────
    slug     = os.path.basename(img_path).replace('_building_mask.png','')
    gt_path  = os.path.join(GT_FINAL_DIR, f'{slug}_gt.npz')

    if os.path.exists(gt_path):
        gt_npz       = np.load(gt_path)
        gt_edge_mask = _make_sub_gt_edge_mask(gt_npz['gt'], free_pixels, sub_ei)
        gt_source    = 'osm'
    else:
        gt_edge_mask = None
        gt_source    = 'test_no_osm'

    # ── 노드 피처 (free pixel 기준, NODE_DIM=7) ────────────────────────────────
    xs = (free_pixels % RES).astype(np.float32) / RES
    ys = (free_pixels // RES).astype(np.float32) / RES

    ridge_sub  = ridge.flatten()[free_pixels]
    dist_n_sub = dist_n.flatten()[free_pixels]

    cluster_indicator = np.zeros(n_free, dtype=np.float32)
    for nd in cluster_sub:
        if 0 <= nd < n_free: cluster_indicator[nd] = 1.0

    d_cluster = _dist_to_px(free_pixels, cluster_nodes, tau=12.0)
    d_gate    = _dist_to_px(free_pixels, gate_nodes,    tau=15.0)

    node_feats = np.stack([ridge_sub, dist_n_sub, xs, ys,
                           cluster_indicator, d_cluster, d_gate], axis=1)

    src_n, dst_n = sub_ei[0].numpy(), sub_ei[1].numpy()
    edge_ridge   = torch.tensor((ridge_sub[src_n]+ridge_sub[dst_n])*0.5,
                                dtype=torch.float32, device=DEVICE)

    return {
        'node_feats':    torch.tensor(node_feats, dtype=torch.float32, device=DEVICE),
        'sub_ei':        sub_ei.to(DEVICE),
        'edge_ridge':    edge_ridge,
        'free_pixels':   free_pixels,
        'pixel_to_node': pixel_to_node,
        'n_free':        n_free,
        'is_bld':        is_bld,
        'ridge':         ridge,
        'cluster_nodes': cluster_nodes,
        'cluster_sub':   cluster_sub,
        'gate_nodes':    gate_nodes,
        'gate_sub':      gate_sub,
        'terminal_nodes':terminals,
        'terminal_sub':  terminal_sub,
        'cluster_groups':cluster_groups,
        'gt_edge_mask':  gt_edge_mask,   # osm=tensor, test_no_osm=None
        'gt_source':     gt_source,
        'poly': poly, 'name': slug, 'W': W, 'H': H,
    }


campuses = []
for img_path in sorted(glob.glob(os.path.join(IMG_DIR,'*_building_mask.png'))):
    txt = _find_txt(img_path)
    if txt:
        try:
            campuses.append(load_campus(img_path, txt))
        except Exception as e:
            print(f'skip {os.path.basename(img_path)}: {e}')

osm_campuses  = [c for c in campuses if c['gt_source'] == 'osm']
test_campuses = [c for c in campuses if c['gt_source'] == 'test_no_osm']
print(f'{len(campuses)}개 캠퍼스 로드 완료  (train+val={len(osm_campuses)}, test={len(test_campuses)})')

for c in osm_campuses[:3]:
    ge    = int(c['gt_edge_mask'].sum())
    ratio = ge / c['sub_ei'].shape[1] * 100
    print(f"  {c['name']}: n_free={c['n_free']}, sub_edges={c['sub_ei'].shape[1]}, "
          f"gt_edges={ge} ({ratio:.2f}%)")

random.seed(42)
random.shuffle(osm_campuses)
n_val = max(1, len(osm_campuses)//7)
val_campuses   = osm_campuses[:n_val]
train_campuses = osm_campuses[n_val:]
print(f'Train: {len(train_campuses)}  Val: {len(val_campuses)}  Test: {len(test_campuses)}')

In [ ]:
# ── GT 시각화 (gt_masks_final 로드 확인) ────────────────────────────────────────
def visualize_gt(campus_name='aalto_university'):
    c = next((x for x in campuses if campus_name in x['name']), None)
    if c is None: print(f'{campus_name} not found'); return
    if c['gt_edge_mask'] is None: print(f'{campus_name}: GT 없음 (test)'); return

    W, H = c['W'], c['H']
    fp   = c['free_pixels']
    ei   = c['sub_ei'].cpu().numpy()
    gt   = c['gt_edge_mask'].cpu().numpy().astype(bool)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')

    for bid, pts in c['poly'].items():
        sc = [(p[0]*RES/W, p[1]*RES/H) for p in pts]
        ax.add_patch(mpatches.Polygon(sc, closed=True,
                     facecolor='#2d3436', edgecolor='#00cec9', lw=0.8, alpha=0.9))

    for i, keep in enumerate(gt):
        if not keep: continue
        s_px = fp[ei[0,i]]; d_px = fp[ei[1,i]]
        y1,x1 = divmod(s_px,RES); y2,x2 = divmod(d_px,RES)
        ax.plot([x1,x2],[y1,y2], color='#ff6b6b', alpha=0.7, lw=1.5, zorder=3)

    for ni, node in enumerate(c['cluster_nodes']):
        y,x = divmod(node,RES)
        ax.scatter(x,y,s=80,c='#74b9ff',edgecolors='white',lw=0.7,zorder=9)
    for gi, node in enumerate(c['gate_nodes']):
        y,x = divmod(node,RES)
        ax.scatter(x,y,s=100,c='#fdcb6e',marker='D',edgecolors='white',lw=0.7,zorder=9)

    ge = int(gt.sum()); ratio = ge/len(gt)*100
    ax.set_xlim(0,RES); ax.set_ylim(RES,0); ax.axis('off')
    ax.set_title(f'{c["name"].replace("_"," ").title()}\n'
                 f'n_free={c["n_free"]}, gt_edges={ge} ({ratio:.2f}%)',
                 color='white', fontsize=11)
    plt.tight_layout(); plt.show(); plt.close()

print(f'train+val GT 캠퍼스: {len(osm_campuses)}개')
for c in osm_campuses[:3]:
    visualize_gt(c['name'])

In [ ]:
# ── 모델: 서브그래프 GAT (v25 edge classifier 구조 동일, 노드 수만 가변) ───────────
class MHGATLayer(nn.Module):
    def __init__(self, in_dim, out_dim, heads=4, dropout=0.0):
        super().__init__()
        assert out_dim % heads == 0
        self.heads=heads; self.dh=out_dim//heads
        self.W    = nn.Linear(in_dim, out_dim, bias=False)
        self.a_s  = nn.Parameter(torch.empty(heads,self.dh))
        self.a_d  = nn.Parameter(torch.empty(heads,self.dh))
        self.norm = nn.LayerNorm(out_dim)
        self.skip = nn.Linear(in_dim,out_dim,bias=False) if in_dim!=out_dim else nn.Identity()
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.W.weight,gain=1.414)
        nn.init.xavier_normal_(self.a_s.unsqueeze(0))
        nn.init.xavier_normal_(self.a_d.unsqueeze(0))

    def forward(self, x, ei):
        n=x.size(0); src,dst=ei[0],ei[1]
        h=self.W(x).view(n,self.heads,self.dh)
        e=F.leaky_relu((h[src]*self.a_s).sum(-1)+(h[dst]*self.a_d).sum(-1),0.2)
        e_exp=torch.exp(e-e.max())
        denom=torch.zeros(n,self.heads,device=x.device)
        denom.scatter_add_(0,src.unsqueeze(1).expand(-1,self.heads),e_exp)
        alpha=self.drop(e_exp/(denom[src]+1e-8))
        msg=alpha.unsqueeze(-1)*h[dst]
        agg=torch.zeros(n,self.heads,self.dh,device=x.device)
        agg.scatter_add_(0,src[:,None,None].expand_as(msg),msg)
        return self.norm(F.elu(agg.view(n,-1))+self.skip(x))


class SubgraphGAT(nn.Module):
    """v25 CampusGAT와 동일 구조. gate_head만 제거 (서브그래프에서 boundary가 별도 없음)."""
    def __init__(self):
        super().__init__()
        self.gat1 = MHGATLayer(NODE_DIM, 64, GAT_HEADS, ATTN_DROPOUT)
        self.gat2 = MHGATLayer(64,       64, GAT_HEADS, ATTN_DROPOUT)
        self.gat3 = MHGATLayer(64,       64, GAT_HEADS, ATTN_DROPOUT)
        # v25와 동일: h_s, h_d, |h_s-h_d|, h_s*h_d + raw 2개
        self.edge_head = nn.Sequential(
            nn.Linear(64*4+2, 128), nn.ReLU(), nn.Dropout(0.08),
            nn.Linear(128, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        for m in self.modules():
            if isinstance(m,nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)
        nn.init.constant_(self.edge_head[-1].bias, -3.0)

    def forward(self, feats, ei):
        h1=self.gat1(feats,ei); h2=self.gat2(h1,ei); h3=self.gat3(h2,ei)
        src,dst=ei[0],ei[1]
        hs,hd=h3[src],h3[dst]
        raw_edge=torch.stack([
            (feats[src,0]+feats[dst,0])*0.5,  # ridge
            (src!=dst).float(),               # self-loop 여부 (서브그래프엔 없지만 호환)
        ],dim=1)
        edge_feat=torch.cat([hs,hd,torch.abs(hs-hd),hs*hd,raw_edge],dim=1)
        edge_logits=self.edge_head(edge_feat).squeeze(-1).float()
        edge_logits=torch.nan_to_num(edge_logits,nan=0.,posinf=20.,neginf=-20.).clamp(-20.,20.)
        ew=torch.sigmoid(edge_logits)
        return ew, edge_logits


model = SubgraphGAT().to(DEVICE)
print(f'파라미터: {sum(p.numel() for p in model.parameters()):,}개')

c0 = campuses[0]
with torch.no_grad():
    ew_t, _ = model(c0['node_feats'], c0['sub_ei'])
print(f'forward ok: n_free={c0["n_free"]}, sub_edges={c0["sub_ei"].shape[1]}, ew={ew_t.shape}')

In [ ]:
# ── 손실 함수 (v25 그대로) ─────────────────────────────────────────────────────
def trunk_bce_loss(edge_logits, gt_edge_mask):
    logits = torch.nan_to_num(edge_logits.float(),nan=0.,posinf=20.,neginf=-20.).clamp(-20.,20.)
    target = gt_edge_mask.float()
    pos = target.sum()
    if pos <= 0:
        return F.binary_cross_entropy_with_logits(logits, target)
    neg = target.numel() - pos
    pos_weight = torch.clamp(neg/(pos+1e-6), min=1., max=80.).detach()
    return F.binary_cross_entropy_with_logits(logits, target, pos_weight=pos_weight)

def soft_dice_loss(ew, gt_edge_mask):
    pred   = ew.float().clamp(0.,1.)
    target = gt_edge_mask.float()
    inter  = (pred*target).sum()
    return 1.0 - (2.*inter+1.)/(pred.sum()+target.sum()+1.)

def false_positive_ridge_loss(ew, gt_edge_mask, edge_ridge):
    fp = gt_edge_mask < 0.5
    if fp.sum() == 0: return ew.sum()*0.
    pred = ew[fp].float().clamp(0.,1.)
    return (pred*(1.-edge_ridge[fp]).clamp(0.2,1.)).mean()

def loss_terms(ew, edge_logits, c, scale):
    bce  = trunk_bce_loss(edge_logits, c['gt_edge_mask'])
    dice = soft_dice_loss(ew, c['gt_edge_mask'])
    fp   = false_positive_ridge_loss(ew, c['gt_edge_mask'], c['edge_ridge']) * (0.2+0.8*scale)
    sparse = ew.float().mean() * 0.03

    with torch.no_grad():
        pred = ew > EDGE_THR
        gt   = c['gt_edge_mask'] > 0
        tp   = (pred & gt).sum().float()
        precision = tp / (pred.sum().float()+1e-6)
        recall    = tp / (gt.sum().float()+1e-6)

    return {
        'bce':bce,'dice':dice,'fp':fp,'sparse':sparse,
        'precision':precision.detach(),'recall':recall.detach(),
    }

def loss_fn(ew, edge_logits, c, scale):
    t = loss_terms(ew, edge_logits, c, scale)
    return torch.nan_to_num(t['bce']+0.8*t['dice']+t['fp']+t['sparse'],nan=0.,posinf=100.,neginf=0.)

In [ ]:
# ── 학습 ────────────────────────────────────────────────────────────────────────
opt    = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sch    = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=5e-6)
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
n = len(train_campuses); loss_curve=[]; val_curve=[]

for ep in range(1, EPOCHS+1):
    model.train(); scale=min(1.0,ep/WARMUP_EPOCHS); total=0.; opt.zero_grad()
    do_debug = (ep%10==0 or ep==1); acc_terms=None

    for i, c in enumerate(train_campuses):
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            ew, edge_logits = model(c['node_feats'], c['sub_ei'])
            loss = loss_fn(ew, edge_logits, c, scale) / n
        if not torch.isfinite(loss): loss = loss*0.
        scaler.scale(loss).backward(); total += loss.item()*n

        if do_debug:
            with torch.no_grad():
                terms={k:float(v.detach().cpu()) for k,v in loss_terms(ew,edge_logits,c,scale).items()}
                if acc_terms is None: acc_terms=terms.copy()
                else:
                    for k in terms: acc_terms[k]+=terms[k]

        if (i+1)%ACCUM_STEPS==0 or (i+1)==n:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),2.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    sch.step(); loss_curve.append(total/n)

    model.eval()
    with torch.no_grad():
        val_total=0.
        for c in val_campuses:
            with torch.amp.autocast('cuda',enabled=torch.cuda.is_available()):
                ew,edge_logits=model(c['node_feats'],c['sub_ei'])
                val_total+=loss_fn(ew,edge_logits,c,scale).item()
        val_curve.append(val_total/len(val_campuses))

    torch.save({'epoch':ep,'model':model.state_dict(),'loss':total/n},CKPT)

    msg=f'[{ep:4d}/{EPOCHS}] loss={total/n:.4f}  val={val_curve[-1]:.4f}  lr={sch.get_last_lr()[0]:.2e}'
    if acc_terms is not None:
        mean_t={k:v/n for k,v in acc_terms.items()}
        msg+='  | '+' '.join(f'{k}={v:.3f}' for k,v in mean_t.items())
    print(msg)

print(f'완료: {CKPT}')

In [ ]:
# ── 시각화 ────────────────────────────────────────────────────────────────────
if os.path.exists(CKPT):
    ck=torch.load(CKPT,map_location=DEVICE)
    model.load_state_dict(ck['model'] if isinstance(ck,dict) else ck)
    print(f'체크포인트 로드: epoch={ck.get("epoch","?") if isinstance(ck,dict) else "?"}')

if loss_curve:
    plt.figure(figsize=(9,3))
    plt.plot(loss_curve,label='train')
    if val_curve: plt.plot(val_curve,label='val')
    plt.yscale('log'); plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


def _prune(selected, terminal_sub, n_free, sub_ei_np):
    selected=selected.copy()
    term_set=set(int(t) for t in terminal_sub)
    src,dst=sub_ei_np[0],sub_ei_np[1]
    for _ in range(PRUNE_ITERS):
        deg=np.zeros(n_free,dtype=np.int32)
        for i,keep in enumerate(selected):
            if not keep: continue
            deg[src[i]]+=1; deg[dst[i]]+=1
        rm=np.zeros_like(selected,dtype=bool); changed=False
        for i,keep in enumerate(selected):
            if not keep: continue
            s,d=src[i],dst[i]
            if (deg[s]<=1 and s not in term_set) or (deg[d]<=1 and d not in term_set):
                rm[i]=True; changed=True
        selected[rm]=False
        if not changed: break
    return selected

def _largest_cc(selected, n_free, sub_ei_np):
    src,dst=sub_ei_np[0],sub_ei_np[1]
    adj={}
    for i,keep in enumerate(selected):
        if not keep: continue
        adj.setdefault(src[i],[]).append(dst[i]); adj.setdefault(dst[i],[]).append(src[i])
    if not adj: return selected
    seen=[False]*n_free; comps=[]
    for v in adj:
        if seen[v]: continue
        stack=[v]; seen[v]=True; comp=[]
        while stack:
            u=stack.pop(); comp.append(u)
            for nb in adj.get(u,[]):
                if not seen[nb]: seen[nb]=True; stack.append(nb)
        comps.append(comp)
    keep_nodes=set(max(comps,key=len))
    out=selected.copy()
    for i,keep in enumerate(selected):
        if not keep: continue
        if src[i] not in keep_nodes or dst[i] not in keep_nodes: out[i]=False
    return out


def plot_campus(c, show_gt=True, use_prune=True):
    model.eval()
    with torch.no_grad():
        ew, _ = model(c['node_feats'], c['sub_ei'])
    ew_np     = ew.cpu().numpy()
    sub_ei_np = c['sub_ei'].cpu().numpy()
    fp        = c['free_pixels']
    W, H      = c['W'], c['H']

    fig,ax=plt.subplots(figsize=(12,12))
    ax.set_facecolor('#0d1117'); fig.patch.set_facecolor('#0d1117')

    for bid,pts in c['poly'].items():
        sc=[(p[0]*RES/W,p[1]*RES/H) for p in pts]
        ax.add_patch(mpatches.Polygon(sc,closed=True,
                     facecolor='#2d3436',edgecolor='#00cec9',lw=0.8,alpha=0.9))
        ax.text(np.mean([p[0] for p in sc]),np.mean([p[1] for p in sc]),
                bid,color='white',ha='center',va='center',fontsize=5.5)

    # gt_edge_mask=None인 test_no_osm 캠퍼스는 GT 표시 스킵
    if show_gt and c['gt_edge_mask'] is not None and c['gt_edge_mask'].sum() > 0:
        gt_em=c['gt_edge_mask'].cpu().numpy().astype(bool)
        for i,keep in enumerate(gt_em):
            if not keep: continue
            s_px=fp[sub_ei_np[0,i]]; d_px=fp[sub_ei_np[1,i]]
            y1,x1=divmod(s_px,RES); y2,x2=divmod(d_px,RES)
            ax.plot([x1,x2],[y1,y2],color='#ff6b6b',alpha=0.3,lw=1.5,zorder=3)

    selected=ew_np>EDGE_THR
    if use_prune:
        selected=_largest_cc(selected,c['n_free'],sub_ei_np)
        selected=_prune(selected,c['terminal_sub'],c['n_free'],sub_ei_np)

    mxw=max(ew_np[selected].max() if selected.any() else EDGE_THR,EDGE_THR+1e-8)
    for i,keep in enumerate(selected):
        if not keep: continue
        w=ew_np[i]
        s_px=fp[sub_ei_np[0,i]]; d_px=fp[sub_ei_np[1,i]]
        y1,x1=divmod(s_px,RES); y2,x2=divmod(d_px,RES)
        ax.plot([x1,x2],[y1,y2],color='#ffd32a',alpha=0.92,
                lw=max(1.2,(w-EDGE_THR)/(mxw-EDGE_THR+1e-8)*5+1),
                solid_capstyle='round',zorder=5)

    for ni,node in enumerate(c['cluster_nodes']):
        y,x=divmod(node,RES)
        ax.scatter(x,y,s=70,c='#74b9ff',edgecolors='white',lw=0.7,zorder=9)
        ax.text(x,y-2,f'C{ni}',color='white',ha='center',fontsize=6)
    for gi,node in enumerate(c['gate_nodes']):
        y,x=divmod(node,RES)
        ax.scatter(x,y,s=90,c='#fdcb6e',marker='D',edgecolors='white',lw=0.7,zorder=9)
        ax.text(x,y-2.5,f'G{gi}',color='#fdcb6e',ha='center',fontsize=6,fontweight='bold')

    n_ge = int(c['gt_edge_mask'].sum()) if c['gt_edge_mask'] is not None else 0
    ax.set_xlim(0,RES); ax.set_ylim(RES,0); ax.axis('off')
    ax.set_title(
        f'V28 Subgraph GNN · {c["name"].replace("_"," ").title()} · '
        f'[{c["gt_source"]}] · n_free={c["n_free"]} · gt_edges={n_ge} · pred={int(selected.sum())}',
        color='white',fontsize=12,pad=8
    )
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR,f'v28_subgraph_{c["name"]}.png'),
                dpi=150,bbox_inches='tight',facecolor='#0d1117')
    plt.show(); plt.close()


TARGET='sungkyunkwan_university'
target=next((c for c in campuses if TARGET in c['name']),None)
if target: plot_campus(target, show_gt=True)

print('\n=== Val set ===')
for c in val_campuses[:3]: plot_campus(c, show_gt=True)

print('\n=== Test set (GT 없음) ===')
for c in test_campuses[:5]: plot_campus(c, show_gt=False)